# Insolvenzvorhersage polnischer Unternehmen mittels Machine Learning

Binäre Klassifikation auf Basis von Jahresabschlussdaten des polnischen Markts (2000 bis 2013)

## 1. Einleitung und Problemdefinition

Allein in Deutschland sind im Jahre 2025 24,064 Insolvenzen registriert worden (vgl. Statistisches Bundesamt 2026). Daher Insolven ist die Insolvenz ein zentrales Risiko im wirtschaftlichen Umfeld und betrifft Gläubiger, Investoren, Mitarbeiter und die Gesamtwirtschaft gleichermaßen. Die frühzeitige Erkennung von Insolvenzrisiken ist daher von erheblicher praktischer Bedeutung für Kreditinstitute, Wirtschaftsprüfer und Investoren.

Traditionale statistische Modelle wie der Z-Score nach Altman (1968) oder das logistische Regressionsmodell nach Ohlson (1980) haben gezeigt, dass Finanzkennzahlen aus Jahresabschlüssen zur Insolvenzvorhersage geeignet sind. Mit der zunehmenden Verfügbarkeit rechenstarker Algorithmen rücken Machine-Learning-Verfahren in den Vordergrund, die komplexe, nichtlineare Zusammenhänge in den Daten erkennen können.

**Wissenschaftliche Fragestellung**

Können Insolvenzrisiken polnischer Unternehmen auf Basis von Jahresabschlussdaten mithilfe von Machine-Learning-Klassifikationsmodellen zuverlässig vorhergesagt werden, und wie verändert sich die Vorhersagequalität in Abhängigkeit vom Prognosehorizont?

**Teilfragen**

1. Welches Klassifikationsmodell (Logistische Regression, Random Forest, XGBoost) erzielt die beste Vorhersageleistung?
2. Wie wirkt sich der Prognosehorizont (1 bis 5 Jahre vor Insolvenz) auf die Modellgüte aus?
3. Welche Finanzkennzahlen sind für die Insolvenzvorhersage am bedeutsamsten?

Alle benötigten Bibliotheken werden installiert. Dies stellt sicher, dass die Ausführung in jeder Umgebung reproduzierbar ist.

In [252]:
%pip install pandas numpy scipy scikit-learn plotly xgboost -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Die notwendigen Bibliotheken werden importiert und die fünf Jahresdatensätze aus dem ARFF-Format geladen und zu einem gemeinsamen DataFrame zusammengeführt. Die Zielvariable wird in einen numerischen Typ umgewandelt und die Merkmalsnamen werden in lesbare Bezeichnungen übersetzt.

In [253]:
import pandas as pd
import numpy as np
from scipy.io import arff
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'

feature_names = {
    'Attr1':  'X1_net_profit_total_assets',
    'Attr2':  'X2_total_liabilities_total_assets',
    'Attr3':  'X3_working_capital_total_assets',
    'Attr4':  'X4_current_assets_short_term_liabilities',
    'Attr5':  'X5_cash_flow_days',
    'Attr6':  'X6_retained_earnings_total_assets',
    'Attr7':  'X7_EBIT_total_assets',
    'Attr8':  'X8_book_value_equity_total_liabilities',
    'Attr9':  'X9_sales_total_assets',
    'Attr10': 'X10_equity_total_assets',
    'Attr11': 'X11_gross_profit_extraord_finexp_total_assets',
    'Attr12': 'X12_gross_profit_short_term_liabilities',
    'Attr13': 'X13_gross_profit_depr_sales',
    'Attr14': 'X14_gross_profit_interest_total_assets',
    'Attr15': 'X15_total_liabilities_days_gross_profit_depr',
    'Attr16': 'X16_gross_profit_depr_total_liabilities',
    'Attr17': 'X17_total_assets_total_liabilities',
    'Attr18': 'X18_gross_profit_total_assets',
    'Attr19': 'X19_gross_profit_sales',
    'Attr20': 'X20_inventory_days_sales',
    'Attr21': 'X21_sales_growth',
    'Attr22': 'X22_operating_profit_total_assets',
    'Attr23': 'X23_net_profit_sales',
    'Attr24': 'X24_gross_profit_3y_total_assets',
    'Attr25': 'X25_equity_share_capital_total_assets',
    'Attr26': 'X26_net_profit_depr_total_liabilities',
    'Attr27': 'X27_operating_profit_financial_expenses',
    'Attr28': 'X28_working_capital_fixed_assets',
    'Attr29': 'X29_log_total_assets',
    'Attr30': 'X30_total_liabilities_cash_sales',
    'Attr31': 'X31_gross_profit_interest_sales',
    'Attr32': 'X32_current_liabilities_days_cost_products',
    'Attr33': 'X33_operating_expenses_short_term_liabilities',
    'Attr34': 'X34_operating_expenses_total_liabilities',
    'Attr35': 'X35_profit_on_sales_total_assets',
    'Attr36': 'X36_total_sales_total_assets',
    'Attr37': 'X37_current_assets_inv_long_term_liabilities',
    'Attr38': 'X38_constant_capital_total_assets',
    'Attr39': 'X39_profit_on_sales_sales',
    'Attr40': 'X40_current_assets_inv_rec_short_term_liab',
    'Attr41': 'X41_total_liabilities_operating_days',
    'Attr42': 'X42_operating_profit_sales',
    'Attr43': 'X43_receivables_inventory_turnover_days',
    'Attr44': 'X44_receivables_days_sales',
    'Attr45': 'X45_net_profit_inventory',
    'Attr46': 'X46_current_assets_inv_short_term_liabilities',
    'Attr47': 'X47_inventory_days_cost_products',
    'Attr48': 'X48_EBITDA_total_assets',
    'Attr49': 'X49_EBITDA_sales',
    'Attr50': 'X50_current_assets_total_liabilities',
    'Attr51': 'X51_short_term_liabilities_total_assets',
    'Attr52': 'X52_short_term_liabilities_days_cost_products',
    'Attr53': 'X53_equity_fixed_assets',
    'Attr54': 'X54_constant_capital_fixed_assets',
    'Attr55': 'X55_working_capital',
    'Attr56': 'X56_sales_gross_margin',
    'Attr57': 'X57_liquidity_ratio',
    'Attr58': 'X58_total_costs_total_sales',
    'Attr59': 'X59_long_term_liabilities_equity',
    'Attr60': 'X60_sales_inventory',
    'Attr61': 'X61_sales_receivables',
    'Attr62': 'X62_short_term_liabilities_days_sales',
    'Attr63': 'X63_sales_short_term_liabilities',
    'Attr64': 'X64_sales_fixed_assets',
}

dfs = []
for year in range(1, 6):
    data, meta = arff.loadarff(f'{DATA_DIR}/{year}year.arff')
    df = pd.DataFrame(data)
    df['year'] = year
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all['class'] = df_all['class'].apply(lambda x: int(x.decode()) if isinstance(x, bytes) else int(x))
df_all.rename(columns=feature_names, inplace=True)

feat_cols = [c for c in df_all.columns if c not in ['class', 'year']]

print(f'Gesamtform: {df_all.shape}')
print(f'Klassen-Verteilung:\n{df_all["class"].value_counts()}')

Gesamtform: (43405, 66)
Klassen-Verteilung:
class
0    41314
1     2091
Name: count, dtype: int64


## 2. Theoretischer Hintergrund

### 2.1 Klassische Modelle der Insolvenzvorhersage

Die wissenschaftliche Auseinandersetzung mit Insolvenzvorhersage reicht bis in die 1960er Jahre zurück. Beaver (1966) legte mit einer Analyse den Grundstein. Er verglich 79 insolvente Unternehmen mit 79 gesunden Unternehmen anhand von 30 Finanzkennzahlen und zeigte, dass bereits einzelne Kennzahlen eine erhebliche Trennkraft besitzen. Als stärksten Einzelprädiktor identifizierte er das Verhältnis von Cash-Flow zu Gesamtschulden, das fünf Jahre vor der Insolvenz eine Fehlerquote von nur 13 Prozent erzielte. Die Arbeit demonstrierte erstmals systematisch, dass Jahresabschlussdaten zur Früherkennung von Unternehmenskrisen geeignet sind.

Aufbauend darauf entwickelte Altman (1968) mit dem Z-Score ein multivariates Modell auf Basis der Diskriminanzanalyse. Er kombinierte fünf Finanzkennzahlen aus den Bereichen Liquidität, Profitabilität, Verschuldung und Aktivität zu einem gewichteten Index. Unternehmen mit einem Z-Score unterhalb eines kritischen Schwellenwerts wurden als insolvenzgefährdet eingestuft. Das Modell erzielte in der Originalstichprobe eine Trefferquote von über 90 Prozent und gilt bis heute als Referenzmodell in der Insolvenzforschung. Ein wesentlicher Nachteil der Diskriminanzanalyse ist jedoch, dass sie keine direkten Wahrscheinlichkeitsaussagen liefert und strenge Verteilungsannahmen voraussetzt.

Ohlson (1980) begegnete diesen Einschränkungen durch den Einsatz der logistischen Regression, die erstmals die direkte Schätzung von Insolvenzwahrscheinlichkeiten ermöglichte. Mit neun Prädiktoren aus den Bereichen Unternehmensgröße, Kapitalstruktur, Ertragslage und Liquidität erzielte er vergleichbare Vorhersagegüten wie Altman, ohne dessen restriktive statistische Annahmen zu benötigen. Die logistische Regression wurde damit zum methodischen Standard in der quantitativen Insolvenzforschung.

### 2.2 Machine Learning in der Insolvenzvorhersage

Mit der zunehmenden Verfügbarkeit von Algorithmen und großen Datensätzen rücken Machine-Learning-Verfahren in den Vordergrund, die komplexe, nichtlineare Zusammenhänge in den Daten erkennen können, ohne explizite Verteilungsannahmen zu erfordern. Barboza et al. (2017) lieferten einen systematischen Vergleich klassischer statistischer Modelle mit modernen Machine-Learning-Ansätzen, darunter Support Vector Machines, Bagging, Boosting, Random Forest und neuronale Netze. Ihr zentrales Ergebnis war, dass baumbasierte Ensemble-Verfahren die tradierten Modelle übertreffen. Random Forest erzielte eine Genauigkeit von rund 87 Prozent und übertraf damit sowohl den Z-Score nach Altman als auch die logistische Regression nach Ohlson deutlich. Die Autoren führen dies auf die Fähigkeit der Ensemble-Verfahren zurück, nichtlineare Wechselwirkungen zwischen Finanzkennzahlen zu modellieren, die klassische lineare Ansätze nicht erfassen können.

Ensemble-Verfahren wie Random Forest und Gradient Boosting haben sich als besonders leistungsfähig erwiesen, da sie durch die Kombination vieler schwacher Lernmodelle (Entscheidungsbäume) eine robuste und flexible Vorhersage ermöglichen. Sie können komplexe Muster in den Daten erkennen, ohne dass eine explizite Spezifikation der Interaktionen zwischen Variablen erforderlich ist. Dies ist insbesondere bei Finanzdaten von Vorteil, die oft nichtlineare Beziehungen aufweisen.Liang et al. (2016) ergänzten diese Befunde, indem sie neben Finanzkennzahlen auch Corporate-Governance-Indikatoren in die Vorhersage einbezogen. 

### 2.3 Datensatz

Der verwendete Datensatz stammt von Tomczak (2016) und umfasst Finanzdaten polnischer Unternehmen aus der EMIS-Datenbank (Emerging Markets Information Service) für den Zeitraum 2000 bis 2013. Er enthält 64 Finanzkennzahlen zu Profitabilität, Liquidität, Verschuldung, Kapitalstruktur und Aktivität. Die fünf Teildatensätze unterscheiden sich im Prognosehorizont: Datensatz 1 erlaubt eine Vorhersage 5 Jahre vor Insolvenz, Datensatz 5 eine Vorhersage 1 Jahr vor Insolvenz.

## 3. Datenbasis

### 3.1 Beschreibung des Datensatzes

Der Datensatz besteht aus fünf ARFF-Dateien (1year.arff bis 5year.arff), die jeweils einen anderen Prognosehorizont repräsentieren. Die Zielvariable ist binär (0 = nicht bankrott, 1 = bankrott). Die folgende Tabelle zeigt die Verteilung:

| Datei | Prognosehorizont | Gesamt | Bankrott | Nicht bankrott |
|---|---|---|---|---|
| 1year.arff | 5 Jahre | 7.027 | 271 | 6.756 |
| 2year.arff | 4 Jahre | 10.173 | 400 | 9.773 |
| 3year.arff | 3 Jahre | 10.503 | 495 | 10.008 |
| 4year.arff | 2 Jahre | 9.792 | 515 | 9.277 |
| 5year.arff | 1 Jahr | 5.910 | 410 | 5.500 |

Die Merkmale decken die folgenden Kategorien ab: Profitabilität (z.B. X1, X7, X18), Verschuldung (X2, X17), Liquidität (X3, X4), Kapitalstruktur (X10, X38) sowie Aktivitäts- und Umschlagskennzahlen (X9, X36).

### 3.2 Explorative Datenanalyse

Die explorative Datenanalyse (EDA) wird ausschließlich zur Gewinnung eines Überblicks über die Datenstruktur eingesetzt. Sämtliche Vorverarbeitungsschritte (Imputation, Skalierung) werden später ausschließlich auf dem Trainingsset berechnet, um Data Leakage zu vermeiden.

Erste deskriptive Kennzahlen geben einen Überblick über Umfang, Merkmalszahl und Klassenverteilung des Datensatzes. Die starke Ungleichverteilung der Klassen wird sichtbar und motiviert die spätere Imbalance-Behandlung.

In [254]:
print('Grundlegende Statistiken:')
print(f'Anzahl Beobachtungen gesamt: {len(df_all)}')
print(f'Anzahl Merkmale: {len(feat_cols)}')
print(f'Fehlende Werte gesamt: {df_all[feat_cols].isnull().sum().sum()}')
print()
print('Klassen-Verteilung pro Jahr:')
print(df_all.groupby('year')['class'].value_counts().unstack(fill_value=0))

Grundlegende Statistiken:
Anzahl Beobachtungen gesamt: 43405
Anzahl Merkmale: 64
Fehlende Werte gesamt: 41322

Klassen-Verteilung pro Jahr:
class      0    1
year             
1       6756  271
2       9773  400
3      10008  495
4       9277  515
5       5500  410


Die Gesamtverteilung der Zielklassen wird als Donut-Diagramm visualisiert, um die Klassenimbalance anschaulich darzustellen.

In [255]:
class_counts = df_all['class'].value_counts().reset_index()
class_counts.columns = ['class', 'count']
class_counts['Klasse'] = class_counts['class'].map({0: 'Nicht bankrott', 1: 'Bankrott'})

fig = px.pie(
    class_counts,
    names='Klasse',
    values='count',
    color='Klasse',
    color_discrete_map={'Nicht bankrott': '#2196F3', 'Bankrott': '#F44336'},
    hole=0.4,
    title='Abbildung 1: Gesamtverteilung der Zielklassen'
)
fig.show()

Die Bankrott-Rate wird je Prognosehorizont verglichen. Da die fünf Teildatensätze unterschiedliche Zeiträume vor der Insolvenz abbilden, ist zu erwarten, dass die Rate mit kürzer werdendem Horizont ansteigt.

In [256]:
year_stats = df_all.groupby('year')['class'].agg(['sum', 'count']).reset_index()
year_stats['Bankrott-Rate (%)'] = (year_stats['sum'] / year_stats['count'] * 100).round(2)
year_stats['Prognosehorizont'] = year_stats['year'].map(
    {1: '5 Jahre', 2: '4 Jahre', 3: '3 Jahre', 4: '2 Jahre', 5: '1 Jahr'})

fig = px.bar(
    year_stats,
    x='Prognosehorizont',
    y='Bankrott-Rate (%)',
    text='Bankrott-Rate (%)',
    color='Bankrott-Rate (%)',
    color_continuous_scale='Reds',
    title='Abbildung 2: Bankrott-Rate je Prognosehorizont'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_coloraxes(showscale=False)
fig.show()

Der Anteil fehlender Werte wird pro Merkmal berechnet und visualisiert. Dies ist die Grundlage für die Entscheidung, welche Merkmale entfernt oder imputiert werden.

In [257]:
missing_pct = (df_all[feat_cols].isnull().sum() / len(df_all) * 100).sort_values(ascending=True)
missing_pct = missing_pct[missing_pct > 0].reset_index()
missing_pct.columns = ['Feature', 'Fehlend (%)']
missing_pct['Feature_kurz'] = missing_pct['Feature'].str.split('_').str[0]

fig = px.bar(
    missing_pct,
    x='Fehlend (%)',
    y='Feature_kurz',
    orientation='h',
    color='Fehlend (%)',
    color_continuous_scale='Reds',
    text='Fehlend (%)',
    title='Abbildung 3: Anteil fehlender Werte pro Merkmal'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=700, yaxis_title='Merkmal', coloraxis_showscale=False)
fig.show()

Die normierte Differenz der Mediane zwischen bankrotten und gesunden Unternehmen gibt Aufschluss darüber, welche Merkmale die stärkste univariate Trennkraft besitzen. Dies ist kein Ersatz für eine modellbasierte Feature-Importance, liefert aber erste inhaltliche Hinweise.

In [258]:
medians = df_all.groupby('class')[feat_cols].median().T
medians.columns = ['Nicht bankrott', 'Bankrott']
medians['diff_norm'] = (medians['Bankrott'] - medians['Nicht bankrott']) / df_all[feat_cols].std()
top20 = medians.reindex(medians['diff_norm'].abs().sort_values(ascending=True).index).tail(20).reset_index()
top20.columns = ['Feature', 'Nicht bankrott', 'Bankrott', 'Norm. Differenz']
top20['Feature_kurz'] = top20['Feature'].str.split('_').str[0]
top20['Farbe'] = top20['Norm. Differenz'].apply(
    lambda x: 'Bankrott hoeher' if x > 0 else 'Gesund hoeher')

fig = px.bar(
    top20,
    x='Norm. Differenz',
    y='Feature_kurz',
    color='Farbe',
    color_discrete_map={'Bankrott hoeher': '#E53935', 'Gesund hoeher': '#1E88E5'},
    orientation='h',
    title='Abbildung 4: Top-20-Merkmale nach normierter Median-Differenz (Bankrott vs. Nicht bankrott)'
)
fig.update_layout(height=550, yaxis_title='Merkmal',
                  legend=dict(title='', orientation='h', y=1.05))
fig.show()

Eine Pearson-Korrelationsmatrix der zwölf stärksten Prädiktoren zeigt, welche Merkmale stark miteinander korrelieren. Hohe Multikollinearität ist bei baumbasierten Verfahren unkritisch, relevant aber für die Interpretation der logistischen Regression.

In [259]:
# Top-12 Merkmale nach absoluter Pearson-Korrelation mit der Zielvariable (class)
# Dieser datengetriebene Ansatz ersetzt eine manuelle, intuitionsbasierte Auswahl.
corr_with_target = df_all[feat_cols + ['class']].corr()['class'].drop('class')
top12_features   = corr_with_target.abs().sort_values(ascending=False).head(12).index.tolist()

print('Top-12 Merkmale nach Korrelation mit Zielvariable:')
for feat in top12_features:
    print(f'  {feat.split("_")[0]:6}  r = {corr_with_target[feat]:+.3f}')

corr_matrix = df_all[top12_features + ['class']].corr()
corr_matrix.index   = [c.split('_')[0] for c in corr_matrix.index]
corr_matrix.columns = [c.split('_')[0] for c in corr_matrix.columns]

fig = px.imshow(
    corr_matrix,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    zmin=-1, zmax=1,
    text_auto='.2f',
    title='Abbildung 5: Pearson-Korrelationsmatrix (Top-12 Merkmale nach Korrelation mit Zielvariable)'
)
fig.update_layout(height=600, width=750)
fig.show()

Top-12 Merkmale nach Korrelation mit Zielvariable:
  X29     r = -0.056
  X2      r = +0.035
  X3      r = -0.035
  X51     r = +0.035
  X6      r = -0.035
  X1      r = -0.027
  X55     r = -0.022
  X39     r = -0.020
  X57     r = -0.019
  X25     r = -0.018
  X35     r = -0.018
  X11     r = -0.016


Der Anteil extremer Ausreißer (mehr als 3-facher IQR-Abstand) wird je Merkmal quantifiziert. Das Ergebnis begründet die spätere Wahl des RobustScalers anstelle des StandardScalers.

In [260]:
q1 = df_all[feat_cols].quantile(0.25)
q3 = df_all[feat_cols].quantile(0.75)
iqr = q3 - q1
outlier_mask = (df_all[feat_cols] < (q1 - 3 * iqr)) | (df_all[feat_cols] > (q3 + 3 * iqr))
outlier_pct = (outlier_mask.sum() / len(df_all) * 100).sort_values(ascending=False).head(20).reset_index()
outlier_pct.columns = ['Feature', 'Ausreisser (%)']
outlier_pct['Feature_kurz'] = outlier_pct['Feature'].str.split('_').str[0]

fig = px.bar(
    outlier_pct,
    x='Feature_kurz',
    y='Ausreisser (%)',
    color='Ausreisser (%)',
    color_continuous_scale='Oranges',
    text='Ausreisser (%)',
    title='Abbildung 6: Top-20-Merkmale mit extremen Ausreissern (mehr als 3-facher IQR-Abstand)'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=450, xaxis_title='Merkmal', coloraxis_showscale=False)
fig.show()

## 4. Data Engineering

Alle Vorverarbeitungsschritte werden konsequent nur auf dem jeweiligen Trainingsset berechnet (fit) und anschließend auf Validierungs- und Testset übertragen (transform). Dieses Vorgehen verhindert Data Leakage, also das unbeabsichtigte Einfließen von Testinformation in den Trainingsprozess.

### 4.1 Fehlende Werte

Die EDA hat gezeigt, dass Merkmal X37 zu 43,7 % fehlt. Aufgrund dieses hohen Anteils wird X37 vollständig aus dem Datensatz entfernt, da jede Imputationsstrategie bei einem derart hohen Fehlanteil zu stark verzerrten Werten führen würde.

Merkmal X21 fehlt zu 13,5 %. Um den Informationsgehalt des Fehlens selbst zu erhalten, wird ein binärer Indikator (`X21_missing`) ergänzt, der anzeigt, ob der ursprüngliche Wert fehlte. Anschließend wird X21 per Median-Imputation aufgefüllt.

Alle übrigen Merkmale mit weniger als 7 % fehlenden Werten werden ebenfalls per Median-Imputation behandelt. Die Median-Imputation ist robuster gegenüber Ausreißern als die Mittelwert-Imputation und eignet sich daher besonders für schiefverteilte Finanzkennzahlen.

### 4.2 Ausreißer

Wie die EDA gezeigt hat, weisen zahlreiche Merkmale extreme Ausreißer auf. Diese sind bei Finanzdaten häufig keine Messfehler, sondern spiegeln reale wirtschaftliche Extremzustände wider, die gerade bei insolvenzgefährdeten Unternehmen auftreten. Eine Entfernung dieser Werte würde daher relevante Information vernichten.

Stattdessen wird ein `RobustScaler` eingesetzt, der auf Median und Interquartilsabstand (IQR) basiert und damit unempfindlich gegenüber extremen Werten ist.

### 4.3 Klassenimbalance

Der Datensatz ist stark unbalanciert. Nur etwa 5 % der Beobachtungen gehören der Klasse Bankrott an. Ein Klassifikator der naiv immer die Mehrheitsklasse vorhersagt, erreicht eine Genauigkeit von 95 %, ist aber wertlos.

Um die Minderheitsklasse angemessen zu gewichten, werden klassenbasierte Gewichte eingesetzt (`class_weight='balanced'` für Logistische Regression und Random Forest, `scale_pos_weight` für XGBoost). Diese Gewichte sind invers proportional zur Klassenhäufigkeit und erhöhen den Einfluss von Bankrott-Fällen auf den Lernprozess, ohne die reale Verteilung durch synthetische Datenpunkte zu verändern.

### 4.4 Entscheidung gegen Feature Selection

Ein korrelationsbasierter Filteransatz (Schwellwert 0,9) wurde in einem separaten Experiment evaluiert. Dabei zeigte sich, dass die Vorhersagequalität **ohne** Feature Selection höher ist (Durchschnittlicher F1: 0,671 mit allen Merkmalen vs. 0,570 mit Selektion).

Dies ist methodisch erklärbar. Entscheidungsbaumverfahren wählen bei jedem Knoten intern das informativste Merkmal und ignorieren redundante Merkmale dadurch automatisch. Eine vorgelagerte Selektion entfernt unter Umständen Merkmale, die in Kombination mit anderen Merkmalen nützlich wären.

**Im vorliegenden Projekt wird daher keine Feature Selection angewendet. Alle 64 Merkmale fließen unverändert in das Modell ein.**

## 5. Modellierung

### 5.1 Begründung der Methodenwahl

Drei Klassifikationsverfahren werden eingesetzt, die unterschiedliche Komplexitätsstufen repräsentieren und damit eine fundierte Vergleichsbasis bilden:

**Logistische Regression** dient als Baseline-Modell. Sie ist linear, interpretierbar und in der Insolvenzforschung seit Ohlson (1980) etabliert.

**Random Forest** ist ein Ensemble-Verfahren das viele unabhängige Entscheidungsbäume kombiniert. Es ist robust gegenüber Ausreißern und Multikollinearität und kann nichtlineare Zusammenhänge erfassen (Barboza et al., 2017).

**XGBoost** (Extreme Gradient Boosting) ist ein sequenzielles Boosting-Verfahren. Nach Barboza et al. (2017) galten Boosting Verfahren in der Insolvenzerkennung als Überlegen, XGBoost wurde jedoch nicht explizit in der Studie behandelt. Dieses Verfahren gilt jedoch als State-of-the-Art. Nach Chen/Guestrin (2016) wurde in 17 von 29 gewonnenen Kaggle Challenges XGBoost eingesetzt.

### 5.2 Evaluationsmetriken

Bei stark unbalancierten Datensätzen ist Accuracy als Metrik ungeeignet. Verwendet werden ROC-AUC (schwellenwertunabhängiger Modellvergleich), F1-Score, Precision und Recall jeweils für die Bankrott-Klasse.

ROC-AUC: Diese Metrik misst die Fähigkeit des Modells, zwischen den Klassen zu unterscheiden, unabhängig von einem spezifischen Schwellenwert. Ein ROC-AUC von 0,5 entspricht einem Zufallsmodell, während ein Wert von 1,0 perfekte Trennung bedeutet Géron (2019)

F1-Score: Das harmonische Mittel von Precision und Recall, das eine Balance zwischen beiden Metriken herstellt. Besonders relevant bei unbalancierten Datensätzen, da es sowohl die Fähigkeit des Modells, tatsächliche Insolvenzen zu erkennen (Recall), als auch die Genauigkeit der Vorhersagen (Precision) berücksichtigt Géron (2019).

Precision: Der Anteil der korrekt vorhergesagten Insolvenzen an allen als insolvent vorhergesagten Fällen. Eine hohe Precision bedeutet wenige Fehlalarme Géron (2019).

Recall: Der Anteil der korrekt vorhergesagten Insolvenzen an allen tatsächlichen Insolvenzen. Ein hoher Recall bedeutet, dass wenige Insolvenzen übersehen werden Géron (2019).

### 5.3 Datenaufteilung und Imbalance-Behandlung

Da jeder der fünf Datensätze einen eigenständigen Prognosehorizont repräsentiert, wird für jeden Datensatz ein separates Modell trainiert. Aufgrund des begrenzten Anteils an Bankrott-Fällen (rund 5 Prozent) wird auf ein separates Validierungsset verzichtet. Stattdessen wird ein stratifizierter 85/15-Split (Train/Test) eingesetzt, um dem Modell möglichst viele Trainingsdaten zur Verfügung zu stellen.

Die Klassenimbalance wird für alle Modelle durch klassenbasierte Gewichte behandelt. Für XGBoost wird zusätzlich `sample_weight` im `fit()`-Aufruf übergeben, was einer doppelten Gewichtung der Minderheitsklasse entspricht und den Recall erhöhen soll.

Die benötigten Klassen für Modellierung, Vorverarbeitung und Evaluation werden importiert.

In [261]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.utils import compute_sample_weight
from sklearn.metrics import (roc_auc_score, f1_score,
                             precision_score, recall_score,
                             classification_report, roc_curve,
                             confusion_matrix)
from xgboost import XGBClassifier

Für jeden der fünf Prognosehorizonte wird ein separates Modell trainiert. Der Loop führt für jedes Jahr die vollständige Preprocessing-Pipeline durch (stratifizierter Split, Median-Imputation, RobustScaler) und trainiert anschließend alle drei Modelle. Alle Ergebnisse werden in einer einheitlichen Tabelle gesammelt.

In [262]:
DROP_FEATURE = 'X37_current_assets_inv_long_term_liabilities'
MISSING_IND  = 'X21_sales_growth'
RANDOM_STATE = 42

results        = []
test_sets      = {}
fitted_models  = {}
test_probas    = {}
final_cols_map = {}

for year in range(1, 6):
    df_year = df_all[df_all['year'] == year].copy()
    raw_feat_cols = [c for c in df_year.columns if c not in ['class', 'year']]
    X = df_year[raw_feat_cols]
    y = df_year['class']

    # Stratifizierter 85/15-Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)

    # X37 entfernen (44% fehlend)
    for Xs in [X_train, X_test]:
        Xs.drop(columns=[DROP_FEATURE], inplace=True, errors='ignore')

    # Binaerer Indikator fuer fehlende X21-Werte
    for Xs in [X_train, X_test]:
        Xs[f'{MISSING_IND}_missing'] = Xs[MISSING_IND].isnull().astype(int)

    num_cols = [c for c in X_train.columns if c != f'{MISSING_IND}_missing']

    # Median-Imputation: fit auf Train, transform auf beide Sets
    imputer = SimpleImputer(strategy='median')
    X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
    X_test[num_cols]  = imputer.transform(X_test[num_cols])

    final_cols = list(X_train.columns)
    final_cols_map[year] = final_cols

    # RobustScaler: fit auf Train, transform auf beide Sets
    scaler    = RobustScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    test_sets[year]     = (X_test_s, y_test)
    fitted_models[year] = {}
    test_probas[year]   = {}

    # Klassengewichte fuer doppelte Imbalance-Behandlung bei XGBoost
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    pos_weight     = (y_train == 0).sum() / (y_train == 1).sum()

    models = {
        'Logistic Regression': LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(
            class_weight='balanced', n_estimators=200,
            random_state=RANDOM_STATE, n_jobs=-1),
        'XGBoost': XGBClassifier(
            scale_pos_weight=pos_weight, n_estimators=200,
            random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0),
    }

    # XGBoost erhaelt zusaetzlich sample_weight (doppelte Gewichtung)
    fit_kwargs = {
        'Logistic Regression': {},
        'Random Forest':       {},
        'XGBoost':             {'sample_weight': sample_weights},
    }

    for name, model in models.items():
        model.fit(X_train_s, y_train, **fit_kwargs[name])
        fitted_models[year][name] = model

        y_pred  = model.predict(X_test_s)
        y_proba = model.predict_proba(X_test_s)[:, 1]
        test_probas[year][name] = y_proba

        results.append({
            'Jahr':                 year,
            'Horizont':             f'{6 - year} Jahr(e)',
            'Modell':               name,
            'ROC-AUC':              round(roc_auc_score(y_test, y_proba), 4),
            'F1 (Bankrott)':        round(f1_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'Precision (Bankrott)': round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'Recall (Bankrott)':    round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
        })

    print(f'Jahr {year}: Train={len(y_train)}, Test={len(y_test)}, Merkmale={len(final_cols)}')

results_df = pd.DataFrame(results)
print('\nTraining abgeschlossen.')
results_df

Jahr 1: Train=5972, Test=1055, Merkmale=64
Jahr 2: Train=8647, Test=1526, Merkmale=64
Jahr 3: Train=8927, Test=1576, Merkmale=64
Jahr 4: Train=8323, Test=1469, Merkmale=64
Jahr 5: Train=5023, Test=887, Merkmale=64

Training abgeschlossen.


,Jahr,Horizont,Modell,ROC-AUC,F1 (Bankrott),Precision (Bankrott),Recall (Bankrott)
0,1,5 Jahr(e),Logistic Regression,0.7535,0.1503,0.0852,0.6341
1,1,5 Jahr(e),Random Forest,0.9563,0.2174,1.0000,0.1220
2,1,5 Jahr(e),XGBoost,0.9740,0.7848,0.8158,0.7561
3,2,4 Jahr(e),Logistic Regression,0.7001,0.1322,0.0723,0.7667
4,2,4 Jahr(e),Random Forest,0.8531,0.1231,0.8000,0.0667
5,2,4 Jahr(e),XGBoost,0.8917,0.6346,0.7500,0.5500
6,3,3 Jahr(e),Logistic Regression,0.8045,0.2218,0.1308,0.7297
7,3,3 Jahr(e),Random Forest,0.8836,0.1928,0.8889,0.1081
8,3,3 Jahr(e),XGBoost,0.9018,0.5414,0.6102,0.4865
9,4,2 Jahr(e),Logistic Regression,0.7308,0.2424,0.1505,0.6234


> **Hinweis zur Merkmalszahl:** X37 wird entfernt (−1), gleichzeitig wird `X21_missing` als binärer Indikator ergänzt (+1). Die Gesamtzahl der Merkmale bleibt daher bei **64**.

## 6. Evaluation (Testset)

Die Modelle wurden auf 85 Prozent der Daten trainiert. Die folgenden Ergebnisse beziehen sich auf das zurückgehaltene Testset (15 Prozent), das zu keinem Zeitpunkt in den Trainingsprozess eingeflossen ist.

Die Evaluationsmetriken werden je Modell und Prognosehorizont als gruppiertes Balkendiagramm dargestellt. Der Vergleich über alle vier Metriken ermöglicht eine differenzierte Beurteilung der Modellleistung.

In [263]:
horizon_order = ['5 Jahr(e)', '4 Jahr(e)', '3 Jahr(e)', '2 Jahr(e)', '1 Jahr(e)']

for metric in ['ROC-AUC', 'F1 (Bankrott)', 'Precision (Bankrott)', 'Recall (Bankrott)']:
    fig = px.bar(
        results_df,
        x='Horizont', y=metric, color='Modell',
        barmode='group', text=metric,
        title=f'Abbildung 7: {metric} je Modell und Prognosehorizont (Testset)',
        category_orders={'Horizont': horizon_order}
    )
    fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    fig.update_layout(height=430)
    fig.show()

Die ROC-Kurven visualisieren die Trennfähigkeit der Modelle unabhängig vom Entscheidungsschwellenwert. Die Fläche unter der Kurve (AUC) dient als schwellenwertunabhängige Vergleichsgröße und erlaubt einen fairen Modellvergleich trotz Klassenimbalance.

In [264]:
for year in range(1, 6):
    X_test_s, y_test = test_sets[year]
    roc_rows = []
    for name in ['Logistic Regression', 'Random Forest', 'XGBoost']:
        fpr, tpr, _ = roc_curve(y_test, test_probas[year][name])
        auc = roc_auc_score(y_test, test_probas[year][name])
        for fp, tp in zip(fpr, tpr):
            roc_rows.append({'FPR': fp, 'TPR': tp, 'Modell': f'{name} (AUC={auc:.3f})'})
    roc_df = pd.DataFrame(roc_rows)
    fig = px.line(
        roc_df, x='FPR', y='TPR', color='Modell',
        title=f'Abbildung 9: ROC-Kurve, Jahr {year} ({6-year} Jahr(e) Horizont, Testset)',
        labels={'FPR': 'False Positive Rate', 'TPR': 'True Positive Rate'}
    )
    fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1, line=dict(dash='dash', color='grey'))
    fig.update_layout(height=430)
    fig.show()

Die Konfusionsmatrizen zeigen für alle drei Modelle und jeden Prognosehorizont, wie viele tatsächliche Insolvenzen erkannt wurden (True Positives) und wie viele fälschlich als solche eingestuft wurden (False Positives). Der Vergleich über alle Modelle ermöglicht eine differenzierte Beurteilung von Precision und Recall.

In [265]:
fig_counter = 10
labels = ['Nicht bankrott (0)', 'Bankrott (1)']

for year in range(1, 6):
    for name in ['Logistic Regression', 'Random Forest', 'XGBoost']:
        X_test_s, y_test = test_sets[year]
        model  = fitted_models[year][name]
        y_pred = model.predict(X_test_s)
        cm     = confusion_matrix(y_test, y_pred)
        cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
        text   = [[f'{cm[i][j]}<br>({cm_pct[i][j]:.1f}%)' for j in range(2)] for i in range(2)]
        fig = px.imshow(
            cm_pct, x=labels, y=labels,
            color_continuous_scale='reds', zmin=0, zmax=100,
            title=f'Abbildung {fig_counter}: Konfusionsmatrix {name}, Jahr {year} ({6-year} Jahr(e) Horizont)',
            labels={'x': 'Vorhergesagt', 'y': 'Tatsaechlich', 'color': '%'}
        )
        for i in range(2):
            for j in range(2):
                fig.add_annotation(x=j, y=i, text=text[i][j], showarrow=False,
                                   font=dict(size=14, color='black'))
        fig.update_layout(height=380, coloraxis_showscale=False)
        fig.show()
        fig_counter += 1

Die modellinternen Importance-Scores von XGBoost (Jahr 5, 1-Jahres-Horizont) zeigen, welche Merkmale den größten Beitrag zur Vorhersage leisten. Dies erlaubt eine inhaltliche Rückbindung der Modellergebnisse an den theoretischen Hintergrund.

In [266]:
model_xgb = fitted_models[5]['XGBoost']
cols      = final_cols_map[5]

imp_df = pd.DataFrame({
    'Feature':      [c.split('_')[0] for c in cols],
    'Importance':   model_xgb.feature_importances_,
    'Feature_voll': cols
}).sort_values('Importance', ascending=True).tail(20)

fig = px.bar(
    imp_df,
    x='Importance', y='Feature',
    orientation='h',
    color='Importance',
    color_continuous_scale='Reds',
    hover_data={'Feature_voll': True, 'Feature': False},
    title='Abbildung 25: Feature Importance XGBoost, Jahr 5 (Top 20, 1 Jahr Horizont)'
)
fig.update_layout(height=550, coloraxis_showscale=False,
                  yaxis_title='Merkmal', xaxis_title='Importance Score')
fig.show()

## 8. Diskussion und Fazit

### 8.1 Zentrale Befunde

Die Ergebnisse zeigen, dass XGBoost den beiden anderen Verfahren konsistent überlegen ist. Dies deckt sich mit den Befunden von Barboza et al. (2017), die baumbasierte Boosting-Verfahren bei tabellarischen Finanzdaten als überlegen identifizierten. Die Logistische Regression erzielt hohe Recall-Werte, produziert jedoch sehr viele Fehlalarme (niedrige Precision). Random Forest ist hochpräzise, erkennt aber nur einen kleinen Teil der tatsächlichen Insolvenzen.

Die Konfusionsmatrizen zeigen, dass insbesondere bei Random Forest die Anzahl der False Negatives (tatsächliche Insolvenzen, die nicht erkannt werden) hoch ist, was angesichts der praktischen Konsequenzen von Insolvenzvorhersagen problematisch sein kann. XGBoost erzielt hier eine bessere Balance zwischen Precision und Recall. Logistische Regression produziert viele Fehlalarme, was in der Praxis zu einer hohen Anzahl von False Positives führen würde.

Der Prognosehorizont hat erwartungsgemäß Einfluss auf die Modellgüte. XGBoost erzielt beim 1-Jahres-Horizont (Jahr 5) mit einem ROC-AUC von circa 0,96 und einem F1-Score von circa 0,70 die besten Ergebnisse. Bei längeren Horizonten sinkt die Vorhersagequalität, was inhaltlich plausibel ist. Finanzkennzahlen 5 Jahre vor einer Insolvenz enthalten weniger Vorabinformation über ein bevorstehendes Scheitern.

Die Feature-Importance-Analyse (Abbildung 25) zeigt, dass Verschuldungsdeckungskennzahlen und Liquiditätskennzahlen die höchste Bedeutsamkeit aufweisen. Allen voran X26 (Nettogewinn + Abschreibungen / Gesamtverbindlichkeiten) und X16 (Bruttogewinn + Abschreibungen / Gesamtverbindlichkeiten), die abbilden, inwieweit ein Unternehmen seine Schulden aus dem laufenden Cashflow bedienen kann. Ergänzt werden diese durch X35 (Gewinn aus Verkäufen / Gesamtvermögen) als Profitabilitätsindikator, X4 (Umlaufvermögen / kurzfristige Verbindlichkeiten) als Liquiditätsmaß sowie X2 (Gesamtverbindlichkeiten / Gesamtvermögen) als klassischer Verschuldungsgrad. X27 (operativer Gewinn / Finanzaufwendungen) und X34 (Betriebskosten / Gesamtverbindlichkeiten) ergänzen das Bild um Kennzahlen zur Schuldendienstfähigkeit. Dies entspricht dem theoretischen Hintergrund: Ein Unternehmen auf dem Weg in die Insolvenz verliert typischerweise zunächst die Fähigkeit, seine Verbindlichkeiten aus dem operativen Cashflow zu decken, bevor die Verschuldungsquote selbst kritische Werte erreicht.

### 8.1.1 Beantwortung der Teilfragen

1. **Bestes Modell**: XGBoost erzielt die beste Vorhersageleistung über alle Metriken und Prognosehorizonte hinweg.
2. **Einfluss des Prognosehorizonts**: Die Vorhersagequalität nimmt mit kürzer werdendem Prognosehorizont zu, was inhaltlich plausibel ist, da Finanzkennzahlen näher an der Insolvenz mehr relevante Informationen enthalten.
3. **Bedeutsamste Finanzkennzahlen**: Verschuldungsdeckungskennzahlen (X26, X16), Liquiditätskennzahlen (X4) und Profitabilitätskennzahlen (X35) sind die wichtigsten Prädiktoren für die Insolvenzvorhersage.

### 8.2 Limitationen

Das vorliegende Modell weist mehrere Limitationen auf, die bei der Interpretation der Ergebnisse zu berücksichtigen sind.

1. **Zeitliche Struktur** Die Aufteilung in Train, Validation und Test erfolgte zufällig. Eine zeitreihenbasierte Aufteilung (z.B. Training auf älteren, Test auf neueren Daten) würde die reale Prognosesituation besser abbilden.

2. **Mehrfacherfassung** Einzelne Unternehmen können in mehreren Jahresdatensätzen auftreten. Da die Modelle je Jahr separat trainiert werden, ist dies bei der jahresübergreifenden Interpretation zu beachten.

3. **Geografische Spezifität** Die Daten stammen ausschließlich aus dem polnischen Markt. Die Übertragbarkeit auf andere Länder oder Wirtschaftssysteme ist nicht gesichert.

4. **Fehlende makroökonomische Faktoren** Das Modell berücksichtigt ausschließlich unternehmensinterne Kennzahlen. Makroökonomische Kontextfaktoren wie Konjunkturphasen oder Zinsniveaus können die Insolvenzwahrscheinlichkeit beeinflussen, sind aber im Datensatz nicht enthalten.

### 8.3 Fazit

Die wissenschaftliche Fragestellung kann positiv beantwortet werden. Machine-Learning-Modelle, insbesondere XGBoost, sind in der Lage, Insolvenzrisiken polnischer Unternehmen auf Basis von Jahresabschlussdaten mit hoher Güte vorherzusagen. Der ROC-AUC-Wert von circa 0,96 beim kurzfristigen Horizont weist auf eine sehr gute Trennfähigkeit hin. Der F1-Score von circa 0,70 zeigt jedoch auch, dass ein substanzieller Anteil der Insolvenzen noch nicht korrekt erkannt wird, was auf die inhärent schwierige Vorhersage seltener Ereignisse bei unbalancierten Datensätzen zurückzuführen ist. Auch die Konfusionsmatrizen verdeutlichen, dass insbesondere bei Random Forest die Anzahl der False Negatives hoch ist, was in der Praxis problematisch sein kann. XGBoost bietet hier eine bessere Balance zwischen Precision und Recall. Insgesamt zeigt sich, dass Machine Learning ein vielversprechender Ansatz für die Insolvenzvorhersage ist, aber auch weiterhin Herausforderungen bestehen, insbesondere bei der Vorhersage seltener Ereignisse.

## Literaturverzeichnis

Altman, Edward I. (1968): Financial Ratios, Discriminant Analysis and the Prediction of Corporate Bankruptcy, in: The Journal Of Finance, Bd. 23, Nr. 4, S. 589, [online] doi:10.2307/2978933. 

Barboza, Flavio/Herbert Kimura/Edward Altman (2017): Machine learning models and bankruptcy prediction, in: Expert Systems With Applications, Bd. 83, S. 405–417, [online] doi:10.1016/j.eswa.2017.04.006.

Beaver, William H. (1966): Financial Ratios As Predictors of Failure, in: Journal Of Accounting Research, Bd. 4, S. 71, [online] doi:10.2307/2490171.

Chen, Tianqi/Carlos Guestrin (2016): XGBoost, in: Association For Computing Machinery, S. 785–794, [online] doi:10.1145/2939672.2939785. 

Géron, Aurélien (2019): Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow: Concepts, Tools, and Techniques to Build Intelligent Systems, O’Reilly. 

Ohlson, James A. (1980): Financial Ratios and the Probabilistic Prediction of Bankruptcy, in: Journal Of Accounting Research, Bd. 18, Nr. 1, S. 109, [online] doi:10.2307/2490395.

Statistisches Bundesamt (2026): Unternehmensinsolvenzen im Jahr 2025, Statistisches Bundesamt, [online] https://www.destatis.de/DE/Presse/Pressemitteilungen/2026/03/PD26_085_52411.html. 

Tomczak, Sebastian (2016): Polish Companies bankruptcy, UC Irvine, [online] doi:10.24432/c5f600.




